# Pre-processing
Cleans the raw NASA NEO dataset, engineers features, handles class imbalance, splits data, and saves processed splits for downstream notebooks.

In [13]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

DATASET_PATH = '../Dataset/Raw/neo.csv'
OUTPUT_DIR   = '../Dataset/Processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42

## 1. Load and Validate Raw Data

In [14]:
df = pd.read_csv(DATASET_PATH)
assert df.shape == (90836, 10), f'Unexpected shape: {df.shape}'
assert df.isnull().sum().sum() == 0, 'Unexpected nulls detected'
print(f'Shape: {df.shape}')
print('\norbiting_body value counts:')
print(df['orbiting_body'].value_counts())
print('\nsentry_object value counts:')
print(df['sentry_object'].value_counts())

Shape: (90836, 10)

orbiting_body value counts:
orbiting_body
Earth    90836
Name: count, dtype: int64

sentry_object value counts:
sentry_object
False    90836
Name: count, dtype: int64


## 2. Drop Irrelevant and Leaky Columns
- `id`, `name` — identifiers with no predictive signal
- `orbiting_body` — near-zero variance (all "Earth")
- `sentry_object` — target leakage: NASA sets this flag based on hazard assessment

In [15]:
df.drop(columns=['id', 'name', 'orbiting_body', 'sentry_object'], inplace=True)
print('Remaining columns:', list(df.columns))

Remaining columns: ['est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'absolute_magnitude', 'hazardous']


## 3. Feature Engineering
Log-transforms correct right-skewed distributions. Derived diameter features capture size and estimation uncertainty.

In [16]:
assert (df['est_diameter_min'] > 0).all(), 'est_diameter_min has zero/negative values'

df['diameter_avg']            = (df['est_diameter_min'] + df['est_diameter_max']) / 2
df['diameter_ratio']          = df['est_diameter_max'] / df['est_diameter_min']
df['log_diameter_avg']        = np.log1p(df['diameter_avg'])
df['log_diameter_ratio']      = np.log1p(df['diameter_ratio'])
df['log_relative_velocity']   = np.log1p(df['relative_velocity'])
df['log_miss_distance']       = np.log1p(df['miss_distance'])

df.drop(columns=['est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance'], inplace=True)

assert not df.isnull().any().any(), 'NaN introduced during feature engineering'
assert not np.isinf(df.select_dtypes('number').values).any(), 'Inf introduced during feature engineering'

print('Engineered feature stats:')
df[['diameter_avg','diameter_ratio','log_diameter_avg','log_diameter_ratio',
    'log_relative_velocity','log_miss_distance','absolute_magnitude']].describe()

Engineered feature stats:


,diameter_avg,diameter_ratio,log_diameter_avg,log_diameter_ratio,log_relative_velocity,log_miss_distance,absolute_magnitude
count,90836.000000,9.083600e+04,90836.000000,9.083600e+04,90836.000000,90836.000000,90836.000000
mean,0.206189,2.236068e+00,0.158558,1.174359e+00,10.630184,17.054995,23.527103
std,0.483001,5.101697e-09,0.209846,1.576511e-09,0.581121,1.137469,2.894086
min,0.000985,2.236068e+00,0.000985,1.174359e+00,5.319817,8.816784,9.230000
25%,0.031156,2.236068e+00,0.030681,1.174359e+00,10.261862,16.661049,21.340000
50%,0.078260,2.236068e+00,0.075349,1.174359e+00,10.696279,17.449051,23.700000
75%,0.232029,2.236068e+00,0.208663,1.174359e+00,11.049693,17.850618,25.700000
max,61.311595,2.236068e+00,4.132148,1.174359e+00,12.375778,18.130310,33.200000


## 4. Encode Target Variables

In [17]:
df['hazardous'] = df['hazardous'].astype(int)
print('Class distribution (hazardous):')
print(df['hazardous'].value_counts())
print(f'\nHazardous ratio: {df["hazardous"].mean():.2%}')

Class distribution (hazardous):
hazardous
0    81996
1     8840
Name: count, dtype: int64

Hazardous ratio: 9.73%


## 5. Define Feature Sets

In [18]:
CLF_FEATURES = [
    'log_diameter_avg', 'log_diameter_ratio', 'log_relative_velocity',
    'log_miss_distance', 'absolute_magnitude', 'diameter_avg', 'diameter_ratio'
]
REG_FEATURES = [
    'log_diameter_avg', 'log_diameter_ratio', 'log_relative_velocity',
    'absolute_magnitude', 'diameter_avg', 'diameter_ratio'
]

X_clf = df[CLF_FEATURES]
y_clf = df['hazardous']
X_reg = df[REG_FEATURES]
y_reg = df['log_miss_distance']

print('Classification features:', CLF_FEATURES)
print('Regression features:    ', REG_FEATURES)

Classification features: ['log_diameter_avg', 'log_diameter_ratio', 'log_relative_velocity', 'log_miss_distance', 'absolute_magnitude', 'diameter_avg', 'diameter_ratio']
Regression features:     ['log_diameter_avg', 'log_diameter_ratio', 'log_relative_velocity', 'absolute_magnitude', 'diameter_avg', 'diameter_ratio']


## 6. Stratified 70 / 15 / 15 Train-Validation-Test Split

In [19]:
X_clf_train, X_clf_temp, y_clf_train, y_clf_temp = train_test_split(
    X_clf, y_clf, test_size=0.30, random_state=RANDOM_STATE, stratify=y_clf
)
X_clf_val, X_clf_test, y_clf_val, y_clf_test = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_clf_temp
)

X_reg_train = X_reg.loc[X_clf_train.index]
X_reg_val   = X_reg.loc[X_clf_val.index]
X_reg_test  = X_reg.loc[X_clf_test.index]
y_reg_train = y_reg.loc[X_clf_train.index]
y_reg_val   = y_reg.loc[X_clf_val.index]
y_reg_test  = y_reg.loc[X_clf_test.index]

print(f'Train: {len(X_clf_train):,} | Val: {len(X_clf_val):,} | Test: {len(X_clf_test):,}')
print(f'\nHazardous — Train: {y_clf_train.mean():.2%} | Val: {y_clf_val.mean():.2%} | Test: {y_clf_test.mean():.2%}')

Train: 63,585 | Val: 13,625 | Test: 13,626

Hazardous — Train: 9.73% | Val: 9.73% | Test: 9.73%


## 7. SMOTE — Oversample Minority Class on Training Split Only

In [20]:
smote = SMOTE(random_state=RANDOM_STATE)
X_clf_train_sm, y_clf_train_sm = smote.fit_resample(X_clf_train, y_clf_train)
print(f'Before SMOTE — Train: {len(X_clf_train):,} | Hazardous: {y_clf_train.sum():,}')
print(f'After  SMOTE — Train: {len(X_clf_train_sm):,} | Hazardous: {y_clf_train_sm.sum():,}')

Before SMOTE — Train: 63,585 | Hazardous: 6,188
After  SMOTE — Train: 114,794 | Hazardous: 57,397


## 8. Feature Scaling
Fit `StandardScaler` on training data only. Apply to validation and test using the same fitted scaler.

In [21]:
scaler_clf = StandardScaler()
X_clf_train_sm_scaled = pd.DataFrame(scaler_clf.fit_transform(X_clf_train_sm), columns=CLF_FEATURES)
X_clf_val_scaled      = pd.DataFrame(scaler_clf.transform(X_clf_val),          columns=CLF_FEATURES)
X_clf_test_scaled     = pd.DataFrame(scaler_clf.transform(X_clf_test),         columns=CLF_FEATURES)

scaler_reg = StandardScaler()
X_reg_train_scaled = pd.DataFrame(scaler_reg.fit_transform(X_reg_train), columns=REG_FEATURES)
X_reg_val_scaled   = pd.DataFrame(scaler_reg.transform(X_reg_val),       columns=REG_FEATURES)
X_reg_test_scaled  = pd.DataFrame(scaler_reg.transform(X_reg_test),      columns=REG_FEATURES)

print('Scaling complete.')

Scaling complete.


## 9. Save Processed Data and Scalers

In [22]:
X_clf_train_sm_scaled.to_csv(f'{OUTPUT_DIR}X_train_clf.csv', index=False)
X_clf_val_scaled.to_csv(f'{OUTPUT_DIR}X_val_clf.csv',        index=False)
X_clf_test_scaled.to_csv(f'{OUTPUT_DIR}X_test_clf.csv',      index=False)
pd.Series(y_clf_train_sm).to_csv(f'{OUTPUT_DIR}y_train_clf.csv',            index=False)
y_clf_val.reset_index(drop=True).to_csv(f'{OUTPUT_DIR}y_val_clf.csv',       index=False)
y_clf_test.reset_index(drop=True).to_csv(f'{OUTPUT_DIR}y_test_clf.csv',     index=False)

X_reg_train_scaled.to_csv(f'{OUTPUT_DIR}X_train_reg.csv', index=False)
X_reg_val_scaled.to_csv(f'{OUTPUT_DIR}X_val_reg.csv',     index=False)
X_reg_test_scaled.to_csv(f'{OUTPUT_DIR}X_test_reg.csv',   index=False)
y_reg_train.reset_index(drop=True).to_csv(f'{OUTPUT_DIR}y_train_reg.csv',   index=False)
y_reg_val.reset_index(drop=True).to_csv(f'{OUTPUT_DIR}y_val_reg.csv',       index=False)
y_reg_test.reset_index(drop=True).to_csv(f'{OUTPUT_DIR}y_test_reg.csv',     index=False)

joblib.dump(scaler_clf, f'{OUTPUT_DIR}scaler_clf.joblib')
joblib.dump(scaler_reg, f'{OUTPUT_DIR}scaler_reg.joblib')

print('All processed files saved to:', OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

All processed files saved to: ../Dataset/Processed/
['y_val_reg.csv', 'y_train_reg.csv', 'scaler_reg.joblib', 'X_val_clf.csv', 'X_train_reg.csv', 'X_train_clf.csv', 'X_test_clf.csv', 'y_val_clf.csv', 'y_test_clf.csv', 'y_train_clf.csv', 'y_test_reg.csv', 'X_test_reg.csv', 'X_val_reg.csv', 'scaler_clf.joblib']
